# GPT-5.5 Cookbook\n
\n
This notebook is a practical starting point for building with GPT-5.5 and GPT-5.5 Pro in the OpenAI API.\n
\n
What is covered:\n
- setup and auth\n
- Responses API and Chat Completions usage\n
- framework integration (LangChain)\n
- tool-enabled agent loop\n
- long-context QA pattern\n
- pricing-aware token tracking\n
- GPT-5.5 vs GPT-5.5 Pro task routing\n
- Codex Fast mode tradeoff math\n
- cyber-safety workflow notes\n
\n
Model-specific snapshot (from launch notes):\n
- API models: `gpt-5.5`, `gpt-5.5-pro`\n
- API context window: 1M tokens\n
- Pricing (`gpt-5.5`): $5 / 1M input, $30 / 1M output\n
- Pricing (`gpt-5.5-pro`): $30 / 1M input, $180 / 1M output\n
- Batch and Flex pricing: 50% of standard API rate\n
- Priority processing: 2.5x standard API rate\n
- In Codex: 400K context and optional Fast mode (1.5x token speed for 2.5x cost)\n
\n
Source:\n
- https://openai.com/index/introducing-gpt-5-5/\n

## 0) Setup and Installation

In [ ]:
# Run this once in a fresh environment.\n
# %pip install -U openai langchain langchain-openai tiktoken

In [ ]:
import os\n
from openai import OpenAI\n
\n
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.5")\n
PRO_MODEL = os.getenv("OPENAI_PRO_MODEL", "gpt-5.5-pro")\n
API_KEY = os.getenv("OPENAI_API_KEY")\n
\n
if not API_KEY:\n
    raise ValueError("Set OPENAI_API_KEY before running this notebook.")\n
\n
client = OpenAI(api_key=API_KEY)\n
print(f"Using model: {MODEL}")\n
print(f"Pro model: {PRO_MODEL}")

## 1) Basic Usage with OpenAI API

In [ ]:
resp = client.responses.create(\n
    model=MODEL,\n
    input="List 5 release checks before shipping a backend change to production.",\n
)\n
\n
print(resp.output_text)

In [ ]:
chat = client.chat.completions.create(\n
    model=MODEL,\n
    messages=[\n
        {"role": "system", "content": "You are a concise coding assistant."},\n
        {"role": "user", "content": "Explain idempotency keys with one practical API example."},\n
    ],\n
)\n
\n
print(chat.choices[0].message.content)

## 2) Framework Integration

In [ ]:
from langchain_openai import ChatOpenAI\n
\n
llm = ChatOpenAI(\n
    model=MODEL,\n
    api_key=API_KEY,\n
    temperature=0,\n
)\n
\n
msg = llm.invoke("Draft a rollback plan for a production schema migration.")\n
print(msg.content)

## 3) Building Agents

In [ ]:
import json\n
\n
def dependency_status(name: str):\n
    table = {"postgres": "ok", "redis": "ok", "legacy-cache": "deprecated"}\n
    return {"dependency": name, "status": table.get(name, "unknown")}\n
\n
tools = [\n
    {\n
        "type": "function",\n
        "function": {\n
            "name": "dependency_status",\n
            "description": "Return status for a service dependency.",\n
            "parameters": {\n
                "type": "object",\n
                "properties": {"name": {"type": "string"}},\n
                "required": ["name"],\n
            },\n
        },\n
    }\n
]

In [ ]:
messages = [\n
    {"role": "system", "content": "You are a release assistant. Use tools before deciding."},\n
    {"role": "user", "content": "Can we deploy if a service depends on legacy-cache?"},\n
]\n
\n
first = client.chat.completions.create(\n
    model=MODEL,\n
    messages=messages,\n
    tools=tools,\n
)\n
\n
assistant_message = first.choices[0].message\n
messages.append(assistant_message.model_dump())\n
\n
if assistant_message.tool_calls:\n
    for call in assistant_message.tool_calls:\n
        if call.function.name == "dependency_status":\n
            args = json.loads(call.function.arguments)\n
            result = dependency_status(args["name"])\n
            messages.append({\n
                "role": "tool",\n
                "tool_call_id": call.id,\n
                "content": json.dumps(result),\n
            })\n
\n
final = client.chat.completions.create(model=MODEL, messages=messages)\n
print(final.choices[0].message.content)

## 4) Advanced Applications

In [ ]:
def chunk_text(text: str, chunk_size: int = 4000):\n
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]\n
\n
# Replace with your real large corpus.\n
large_doc = """\n
GPT-5.5 can handle long, multi-part workflows with better persistence and fewer retries.\n
Use chunking and grounded prompting when working with long context inputs.\n
""" * 600\n
\n
chunks = chunk_text(large_doc)\n
context = "\n\n".join(chunks[:60])\n
\n
qa = client.chat.completions.create(\n
    model=MODEL,\n
    messages=[\n
        {"role": "system", "content": "Answer using only the provided context."},\n
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: Which rollout steps reduce outage risk the most?"},\n
    ],\n
)\n
print(qa.choices[0].message.content)

In [ ]:
json_resp = client.chat.completions.create(\n
    model=MODEL,\n
    messages=[\n
        {"role": "system", "content": "Return valid JSON only."},\n
        {"role": "user", "content": "Produce a release plan with fields: risk, owner, mitigation, and rollback_trigger. Return an array with 3 items."},\n
    ],\n
)\n
print(json_resp.choices[0].message.content)

In [ ]:
PRICING = {\n
    "gpt-5.5": {"input_per_m": 5.0, "output_per_m": 30.0},\n
    "gpt-5.5-pro": {"input_per_m": 30.0, "output_per_m": 180.0},\n
}\n
\n
def estimate_cost(model: str, prompt_tokens: int, completion_tokens: int, multiplier: float = 1.0):\n
    rates = PRICING[model]\n
    in_cost = (prompt_tokens / 1_000_000) * rates["input_per_m"]\n
    out_cost = (completion_tokens / 1_000_000) * rates["output_per_m"]\n
    return multiplier * (in_cost + out_cost)\n
\n
usage_demo = client.chat.completions.create(\n
    model=MODEL,\n
    messages=[{"role": "user", "content": "Give 3 concise checks for safe production deploys of AI agents."}],\n
)\n
\n
usage = usage_demo.usage\n
pt = usage.prompt_tokens\n
ct = usage.completion_tokens\n
\n
std_cost = estimate_cost(MODEL, pt, ct)\n
batch_or_flex_cost = estimate_cost(MODEL, pt, ct, multiplier=0.5)\n
priority_cost = estimate_cost(MODEL, pt, ct, multiplier=2.5)\n
\n
print("prompt_tokens:", pt)\n
print("completion_tokens:", ct)\n
print("total_tokens:", usage.total_tokens)\n
print(f"standard_estimated_cost_usd: {std_cost:.6f}")\n
print(f"batch_or_flex_estimated_cost_usd: {batch_or_flex_cost:.6f}")\n
print(f"priority_estimated_cost_usd: {priority_cost:.6f}")

## 5) Model-Specific Workflow: Route Between GPT-5.5 and GPT-5.5 Pro

Use `gpt-5.5` for most high-throughput coding and analysis workloads.\n
Route only the hardest, highest-stakes tasks to `gpt-5.5-pro` where extra accuracy justifies higher cost.

In [ ]:
def pick_model(task: str, risk_level: str = "normal"):\n
    hard_keywords = ["migration", "security", "legal", "compliance", "critical"]\n
    hard = any(k in task.lower() for k in hard_keywords)\n
    if risk_level == "high" or hard:\n
        return PRO_MODEL\n
    return MODEL\n
\n
task = "Review this production security migration plan and identify hidden failure modes."\n
chosen = pick_model(task, risk_level="high")\n
\n
resp = client.responses.create(model=chosen, input=task)\n
print(f"Chosen model: {chosen}")\n
print(resp.output_text)

## 6) Model-Specific Workflow: Codex Fast Mode Tradeoff

Launch notes include Codex Fast mode for GPT-5.5: about 1.5x token speed at 2.5x cost.\n
Use this calculator to decide if lower latency is worth the spend for your workload.

In [ ]:
def codex_fast_mode_estimate(standard_seconds: float, standard_cost_usd: float):\n
    fast_seconds = standard_seconds / 1.5\n
    fast_cost = standard_cost_usd * 2.5\n
    return {\n
        "standard_seconds": standard_seconds,\n
        "standard_cost_usd": standard_cost_usd,\n
        "fast_mode_seconds": fast_seconds,\n
        "fast_mode_cost_usd": fast_cost,\n
    }\n
\n
example = codex_fast_mode_estimate(standard_seconds=180, standard_cost_usd=0.12)\n
for k, v in example.items():\n
    print(f"{k}: {v}")

## 7) Model-Specific Workflow: Cyber-Safety Aware Ops

GPT-5.5 ships with stricter cyber safeguards and a Trusted Access path for verified defensive work.\n
For production teams, add an explicit review gate for security-sensitive automation.

In [ ]:
def cyber_preflight_check(use_case: str, trusted_access_enabled: bool, human_reviewer: str):\n
    return {\n
        "use_case": use_case,\n
        "trusted_access_enabled": trusted_access_enabled,\n
        "human_reviewer": human_reviewer,\n
        "checks": [\n
            "Log all tool actions for audit",\n
            "Block repeated high-risk misuse patterns",\n
            "Require human approval before any external change",\n
            "Run post-action validation tests",\n
        ],\n
    }\n
\n
plan = cyber_preflight_check(\n
    use_case="internal vulnerability triage assistant",\n
    trusted_access_enabled=False,\n
    human_reviewer="security-oncall@company.com",\n
)\n
print(json.dumps(plan, indent=2))

## Model-Specific Notes

- API deployment note: launch post update says GPT-5.5 and GPT-5.5 Pro are available in the API.\n
- GPT-5.5 is tuned for stronger persistence in long-running agentic workflows with better token efficiency than GPT-5.4.\n
- Codex context window is 400K; API context window is 1M.\n
- For most teams, default to `gpt-5.5` and route selectively to `gpt-5.5-pro` for highest-accuracy tasks.\n
- Use Batch/Flex for offline or async heavy jobs and Priority for latency-sensitive workflows.\n
- Validate security-sensitive workflows against your organization policy before production rollout.

---\n
Cookbook done. Replace placeholders with your real prompts, tools, and safety checks, then run each section end-to-end in your target environment.